# Mô hình C: fine-tune BARTpho-syllable (Kaggle / Colab)

Notebook mỏng: mọi logic nằm trong `src/vspell/models/bartpho.py` và `scripts/`. Chạy lần lượt từ trên xuống.

**Chuẩn bị (làm một lần trên máy):**
1. Repo đã push lên GitHub (điền `REPO_URL` ở ô dưới; repo private thì dùng URL có token, hoặc upload repo thành dataset).
2. **Upload dữ liệu thành Kaggle dataset**: tạo dataset mới (vd tên `vspell-data`), upload **mọi** file trong `data/kaggle_upload/`, gồm cả các manifest `*.parts.json` (tạo bằng `python scripts/pack_for_kaggle.py`: nén gz, cắt phần < 9 MB) hoặc các file
   `data/processed/*.jsonl` (`syn_train.jsonl`, `syn_val.jsonl`, `syn_test.jsonl`, `vsec_test.jsonl`, `viwiki_test.jsonl`;
   ablation thêm `syn_train_d05/d12/d25.jsonl`). Dữ liệu không nằm trong git.
   Trong notebook: *Add input* → chọn dataset đó → nó xuất hiện ở `/kaggle/input/vspell-data/`.
   Colab: đặt các file đó lên Google Drive và sửa `DATA_SRC`.
3. Kaggle: *Settings → Accelerator → GPU T4 x2* (2 GPU thì code tự giảm tích lũy gradient để batch hiệu dụng vẫn 32) hoặc P100, *Internet: On* (để tải `vinai/bartpho-syllable`).

**Thời gian ước lượng (fp16):** 150k cặp × 3 epoch ≈ 14k bước; 1 x T4 khoảng 3–4,5 giờ, T4 x2 khoảng 2–3 giờ; P100 (không tăng tốc fp16 nhiều) khoảng 4–5 giờ + ~20–40 phút predict 3 tập test.
Kaggle giới hạn 12 giờ/phiên; checkpoint (chỉ trọng số) lưu mỗi 1000 bước.

In [ ]:
# ==== Sửa các biến này ====
REPO_URL = "https://github.com/Dooty26/vietnamese-spell-correction.git"   # URL repo GitHub
BRANCH   = "main"
COMMIT   = None     # mã commit cố định để tái lập một lần chạy cũ (None = đầu nhánh BRANCH); commit thực tế được ghi vào vspell_config.json
PIN_VERSIONS = True # True: cài đúng requirements-bartpho.txt; False: cài theo khoảng phiên bản (phát triển)
DATA_SRC = "/kaggle/input"   # tìm đệ quy *.jsonl / *.partNN.jsonl(.gz) + *.parts.json; (Colab: "/content/drive/MyDrive/vspell-data")
RUN_MAIN     = True                                 # False: bỏ qua chạy thử + train/predict chính (chỉ chạy ablation)
RUN_ABLATION = False                                # True: train thêm 3 mô hình mật độ nhiễu (50k cặp, 1 epoch)
ABLATION_TAGS = ["d05", "d12", "d25"]               # dùng file syn_train_{tag}.jsonl; v4 (nhiễu nhiều lỗi dấu): ["v4"]
ABLATION_TEST = ["vsec_test"]                       # tập test để predict cho mô hình ablation

In [ ]:
import os, subprocess, shutil, sys, glob
WORK = "/kaggle/working" if os.path.exists("/kaggle/working") else "/content"
REPO = f"{WORK}/repo"
if not os.path.exists(REPO):
    if COMMIT:
        subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO], check=True)
        subprocess.run(["git", "-C", REPO, "checkout", COMMIT], check=True)
    else:
        subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, REPO], check=True)
os.chdir(REPO)
!git log --oneline -1


def run(script, *args):
    """Chạy `python -X utf8 scripts/<script> ...`; lỗi (vd hết bộ nhớ GPU) thì dừng cell ngay,
    không chạy tiếp các lệnh dọn dẹp phía sau như `!python` vẫn làm."""
    cmd = [sys.executable, "-X", "utf8", f"scripts/{script}", *map(str, args)]
    print("$", " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True)


def clean_checkpoints(out_dir):
    """Chỉ xóa checkpoint-* khi best/ đã lưu đủ (trọng số + vspell_config.json)."""
    best = f"{out_dir}/best"
    weights = glob.glob(f"{best}/model.safetensors") + glob.glob(f"{best}/pytorch_model*.bin")
    if not weights or not os.path.exists(f"{best}/vspell_config.json"):
        raise RuntimeError(f"{best} chưa lưu đủ, giữ nguyên các checkpoint-* để resume")
    for d in glob.glob(f"{out_dir}/checkpoint-*"):
        shutil.rmtree(d)
    print("Đã dọn checkpoint-*, giữ", best)

In [ ]:
# Kaggle/Colab đã có torch + CUDA: chỉ cài phần còn thiếu, KHÔNG cài lại torch theo requirements.txt
if PIN_VERSIONS:
    !pip install -q -r requirements-bartpho.txt
else:
    !pip install -q "transformers>=4.56" "datasets>=2.19" "accelerate>=0.30" sentencepiece PyYAML
!pip install -q -e . --no-deps
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Colab + Drive: bỏ comment 2 dòng dưới
# from google.colab import drive; drive.mount("/content/drive")
# Dữ liệu upload dạng `{tên}.partNN.jsonl.gz` + `{tên}.parts.json` (tạo bằng scripts/pack_for_kaggle.py)
# hoặc `{tên}.jsonl` nguyên bản. Script kiểm tra đủ phần theo manifest, ghép qua file tạm rồi mới đổi tên,
# nên lỗi giữa chừng không để lại file thiếu; chạy lại cell này luôn ghép lại từ đầu.
# Bộ upload cũ chưa có *.parts.json: thêm "--allow-no-manifest" (khi đó không phát hiện được thiếu phần cuối).
run("unpack_parts.py", "--src", DATA_SRC, "--dest", "data/processed")
!wc -l data/processed/*.jsonl

## Chạy thử nhanh (~2 phút): 200 cặp, 4 bước
Kiểm tra môi trường trước khi chạy dài. Có thể bỏ qua.

In [ ]:
if RUN_MAIN:
    run("train_bartpho.py", "--limit", 200, "--val-limit", 50, "--max-steps", 4, "--batch-size", 4,
        "--output-dir", "outputs/checkpoints/bartpho_smoke")
    run("predict.py", "--model", "bartpho", "--checkpoint", "outputs/checkpoints/bartpho_smoke/best",
        "--data", "data/processed/vsec_test.jsonl", "--limit", 20, "--output", "/tmp/smoke.jsonl")
    shutil.rmtree("outputs/checkpoints/bartpho_smoke")   # chỉ là chạy thử; giải phóng /kaggle/working (~20 GB)
    !df -h /kaggle/working 2>/dev/null || df -h .

## Train đầy đủ: 150k cặp, 3 epoch
Checkpoint tốt nhất theo loss Syn-val → `outputs/checkpoints/bartpho/best/`. Checkpoint chỉ lưu trọng số (`save_only_model`) để không đầy đĩa; `--resume` chỉ khôi phục trọng số, không khôi phục optimizer.

In [ ]:
if RUN_MAIN:
    run("train_bartpho.py", "--config", "configs/bartpho.yaml", "--train", "data/processed/syn_train.jsonl",
        "--val", "data/processed/syn_val.jsonl", "--output-dir", "outputs/checkpoints/bartpho")
    clean_checkpoints("outputs/checkpoints/bartpho")   # chỉ giữ best/, và chỉ khi best/ đã lưu xong
    !df -h /kaggle/working 2>/dev/null || df -h .

## Predict 3 tập test (tag `bartpho`, beam 4)

In [ ]:
if RUN_MAIN:
    for name in ["syn_test", "vsec_test", "viwiki_test"]:
        run("predict.py", "--model", "bartpho", "--checkpoint", "outputs/checkpoints/bartpho/best",
            "--data", f"data/processed/{name}.jsonl", "--tag", "bartpho")
    run("evaluate.py", "--pred-dir", "outputs/predictions", "--out", "outputs/tables/main_results_bartpho.csv")
    !cat outputs/tables/main_results_bartpho.csv

## (Tùy chọn) Ablation mật độ nhiễu
3 mô hình × 50k cặp × 1 epoch (≈ 1550 bước mỗi mô hình, khoảng 25–35 phút mỗi mô hình trên T4), predict VSEC-test với tag `bartpho_d05/d12/d25`.

Thí nghiệm **v4** (nhiễu nhiều lỗi dấu thanh/dấu phụ, `configs/noise_v4.yaml`) dùng lại ô này: `ABLATION_TAGS = ["v4"]`, `ABLATION_TEST = ["syn_test", "vsec_test", "viwiki_test"]`.


In [ ]:
if RUN_ABLATION:
    for d in ABLATION_TAGS:
        out_dir = f"outputs/checkpoints/bartpho_{d}"
        run("train_bartpho.py", "--train", f"data/processed/syn_train_{d}.jsonl", "--limit", 50000, "--epochs", 1,
            "--eval-steps", 500, "--output-dir", out_dir)
        for t in ABLATION_TEST:
            run("predict.py", "--model", "bartpho", "--checkpoint", f"{out_dir}/best",
                "--data", f"data/processed/{t}.jsonl", "--tag", f"bartpho_{d}")
        clean_checkpoints(out_dir)   # tiết kiệm dung lượng /kaggle/working
        # Colab có thể ngắt giữa chừng: tải kết quả của từng mô hình ngay khi xong
        try:
            from google.colab import files
            for t in ABLATION_TEST:
                files.download(f"outputs/predictions/bartpho_{d}__{t}.jsonl")
        except ImportError:
            pass

## Nén dự đoán để tải về
Tải `bartpho_predictions.zip` (Kaggle: tab *Output*; Colab: ô dưới tự tải), giải nén vào `outputs/predictions/` trên máy.
Checkpoint (~1,6 GB) giữ trên Kaggle/Drive, không đưa vào git.

In [ ]:
!cd outputs/predictions && zip -j {WORK}/bartpho_predictions.zip bartpho*__*.jsonl
if os.path.exists("outputs/checkpoints/bartpho/best/vspell_config.json"):
    !cp outputs/checkpoints/bartpho/best/vspell_config.json {WORK}/bartpho_vspell_config.json
!pip freeze | grep -iE "^(torch|transformers|datasets|accelerate|sentencepiece)==" > {WORK}/bartpho_pip_freeze.txt
!ls -la {WORK}
try:
    from google.colab import files; files.download(f"{WORK}/bartpho_predictions.zip")
except ImportError:
    pass
